In [1]:
!pip install chromadb faiss-cpu sentence-transformers datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fou

## ChromaDB

In [2]:
import chromadb
from chromadb.utils import embedding_functions

# PersistentClient saves to disk — survives kernel restarts
# In-memory alternative: chromadb.Client()
client = chromadb.PersistentClient(path="./chroma_store")

# SentenceTransformer embedding function
# ChromaDB will call this automatically when you add/query text
# You never manually call model.encode() with ChromaDB
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# get_or_create — safe to re-run without duplicating
# hnsw:space tells ChromaDB which distance metric to use
# cosine → good for sentence embeddings
# l2     → euclidean distance
# ip     → inner product / dot product
collection = client.get_or_create_collection(
    name="news_articles",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
docs = [
    "SpaceX successfully launches Starship on orbital test",
    "Python 4.0 releases with major performance improvements",
    "Scientists discover new exoplanet in habitable zone",
    "India wins cricket World Cup in thrilling final",
    "New AI model beats humans at protein folding tasks",
    "Stock market hits all-time high amid tech rally",
    "Breakthrough in battery technology promises 1000km range",
    "Neural interfaces allow paralyzed patients to type with thoughts",
]

metadatas = [
    {"category": "space",   "source": "techcrunch"},
    {"category": "tech",    "source": "python.org"},
    {"category": "science", "source": "nasa"},
    {"category": "sports",  "source": "espn"},
    {"category": "AI",      "source": "arxiv"},
    {"category": "finance", "source": "bloomberg"},
    {"category": "tech",    "source": "ieee"},
    {"category": "science", "source": "nature"},
]

collection.add(
    documents=docs,       # raw text — ChromaDB embeds it using your ef
    metadatas=metadatas,  # any key-value pairs you want to filter on later
    ids=[f"doc_{i}" for i in range(len(docs))]
    # ids are mandatory and must be unique strings
    # used for updates, deletes, and deduplication
)

print(f"Collection size: {collection.count()}")


Collection size: 8


In [4]:
results = collection.query(
    query_texts=["latest developments in artificial intelligence"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
    # distances here = cosine distance (0 = identical, 2 = opposite)
    # NOT similarity — lower is better
)

# results is a dict of lists-of-lists
# outer list = one entry per query (you can batch queries)
# inner list = n_results items
for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    similarity = 1 - dist   # convert cosine distance → cosine similarity
    print(f"[{meta['category']}] score={similarity:.3f} | {doc}")

[AI] score=0.377 | New AI model beats humans at protein folding tasks
[tech] score=0.234 | Python 4.0 releases with major performance improvements
[science] score=0.155 | Neural interfaces allow paralyzed patients to type with thoughts


In [5]:
# Single filter
results = collection.query(
    query_texts=["software and programming"],
    n_results=2,
    where={"category": "tech"}
    # Only searches within documents where category == "tech"
    # The similarity search happens AFTER this filter is applied
)

# Multiple filters with $and
results = collection.query(
    query_texts=["scientific discovery"],
    n_results=2,
    where={
        "$and": [
            {"category": "science"},
            {"source": "nasa"}
        ]
    }
)

# Filter with $in — match any value in a list
results = collection.query(
    query_texts=["technology advances"],
    n_results=3,
    where={"category": {"$in": ["tech", "AI"]}}
)

# Filter with $ne — not equal
results = collection.query(
    query_texts=["technology advances"],
    n_results=3,
    where={"category": {"$ne": "sports"}}
)

for doc in results["documents"][0]:
    print(doc)

Breakthrough in battery technology promises 1000km range
Python 4.0 releases with major performance improvements
Stock market hits all-time high amid tech rally


In [6]:
# Update — overwrite document text and/or metadata by ID
collection.update(
    ids=["doc_0"],
    documents=["SpaceX Starship completes first full orbital flight"],
    metadatas=[{"category": "space", "source": "spacex.com"}]
    # ChromaDB re-embeds the new text automatically
)

# Delete by ID
collection.delete(ids=["doc_5"])
print(f"Size after delete: {collection.count()}")  # 7

# Delete by metadata filter — delete all sports articles
collection.delete(where={"category": "sports"})
print(f"Size after bulk delete: {collection.count()}")  # 6

Size after delete: 7
Size after bulk delete: 6


## FAISS
FAISS is lower level. It only knows about vectors — no text, no metadata. You manage the document store yourself.

In [7]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

docs = [
    "SpaceX successfully launches Starship on orbital test",
    "Python 4.0 releases with major performance improvements",
    "Scientists discover new exoplanet in habitable zone",
    "India wins cricket World Cup in thrilling final",
    "New AI model beats humans at protein folding tasks",
    "Breakthrough in battery technology promises 1000km range",
]

categories = ["space", "tech", "science", "sports", "AI", "tech"]

# model.encode returns float32 numpy array of shape (6, 384)
embeddings = model.encode(docs, show_progress_bar=True)
embeddings = embeddings.astype(np.float32)
# FAISS strictly requires float32 — will throw errors with float64

dim = embeddings.shape[1]  # 384
print(f"Embeddings shape: {embeddings.shape}")  # (6, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (6, 384)


In [8]:
# FlatL2 = brute-force exact search using L2 (euclidean) distance
# No training needed — just add and search
index_flat = faiss.IndexFlatL2(dim)

# add() just appends vectors to an internal array
# FAISS auto-assigns integer IDs: 0, 1, 2, ...
index_flat.add(embeddings)
print(f"Index size: {index_flat.ntotal}")  # 6

# Search
query_text = "AI and machine learning research"
query_vec = model.encode([query_text]).astype(np.float32)
# query_vec shape must be (1, 384) — 2D array even for single query

k = 3  # return top-3 results
D, I = index_flat.search(query_vec, k)
# D = distances array, shape (1, k) — lower = more similar (L2)
# I = indices array, shape (1, k) — which documents matched

print(f"\nQuery: '{query_text}'")
for dist, idx in zip(D[0], I[0]):
    print(f"  [{idx}] dist={dist:.4f} | {docs[idx]}")

Index size: 6

Query: 'AI and machine learning research'
  [4] dist=1.3132 | New AI model beats humans at protein folding tasks
  [2] dist=1.8922 | Scientists discover new exoplanet in habitable zone
  [5] dist=1.9102 | Breakthrough in battery technology promises 1000km range


In [9]:
# Cosine similarity with FAISS
faiss.normalize_L2(embeddings)        # normalizes in-place, modifies array
faiss.normalize_L2(query_vec)

index_cosine = faiss.IndexFlatIP(dim) # IP = inner product
index_cosine.add(embeddings)

D_cos, I_cos = index_cosine.search(query_vec, k)
# Now D values are cosine similarities — higher = more similar
for sim, idx in zip(D_cos[0], I_cos[0]):
    print(f"  [{idx}] cosine_sim={sim:.4f} | {docs[idx]}")

  [4] cosine_sim=0.3434 | New AI model beats humans at protein folding tasks
  [2] cosine_sim=0.0539 | Scientists discover new exoplanet in habitable zone
  [5] cosine_sim=0.0449 | Breakthrough in battery technology promises 1000km range


In [10]:
nlist = 2
# nlist = number of Voronoi cells (clusters)
# Rule of thumb: nlist = sqrt(n_vectors) for large datasets
# With only 6 docs we use 2 — in production use 100-1000

quantizer = faiss.IndexFlatL2(dim)
# quantizer decides how to measure distance between clusters
# always IndexFlatL2 for IVF

index_ivf = faiss.IndexIVFFlat(quantizer, dim, nlist)

# Training — runs k-means to find cluster centroids
# Must happen BEFORE add()
# Needs at least nlist * 39 vectors to train reliably
index_ivf.train(embeddings)
print(f"IVF trained: {index_ivf.is_trained}")  # True

index_ivf.add(embeddings)

# nprobe = how many clusters to search at query time
# nprobe=1 → fastest but may miss results
# nprobe=nlist → same as exact search but slower than FlatL2
index_ivf.nprobe = 1

D_ivf, I_ivf = index_ivf.search(query_vec, k)
for dist, idx in zip(D_ivf[0], I_ivf[0]):
    print(f"  [{idx}] dist={dist:.4f} | {docs[idx]}")

IVF trained: True
  [4] dist=1.3132 | New AI model beats humans at protein folding tasks
  [1] dist=1.9312 | Python 4.0 releases with major performance improvements
  [-1] dist=340282346638528859811704183484516925440.0000 | Breakthrough in battery technology promises 1000km range


In [11]:
# Save index to disk
faiss.write_index(index_flat, "news.index")

# Load it back — in a new session or after restart
loaded_index = faiss.read_index("news.index")
print(f"Loaded index size: {loaded_index.ntotal}")  # 6

# FAISS only stores vectors — you manage text separately
# Build a doc store that mirrors the FAISS integer IDs
doc_store = {
    i: {"text": doc, "category": cat}
    for i, (doc, cat) in enumerate(zip(docs, categories))
}

# Search + retrieve text
query_vec2 = model.encode(["space exploration"]).astype(np.float32)
D2, I2 = loaded_index.search(query_vec2, k=2)

print("\nResults with metadata:")
for idx in I2[0]:
    entry = doc_store[idx]
    print(f"  [{entry['category']}] {entry['text']}")

Loaded index size: 6

Results with metadata:
  [space] SpaceX successfully launches Starship on orbital test
  [AI] New AI model beats humans at protein folding tasks


In [12]:
import time

query_texts = [
    "artificial intelligence research",
    "space exploration missions",
    "cricket sports results",
    "programming languages",
    "medical science breakthroughs",
]

# Benchmark ChromaDB
chroma_times = []
for q in query_texts:
    start = time.perf_counter()
    collection.query(query_texts=[q], n_results=2)
    chroma_times.append((time.perf_counter() - start) * 1000)

# Benchmark FAISS
faiss_times = []
for q in query_texts:
    q_vec = model.encode([q]).astype(np.float32)
    start = time.perf_counter()
    index_flat.search(q_vec, 2)
    faiss_times.append((time.perf_counter() - start) * 1000)

# Note: ChromaDB time includes embedding the query text
# FAISS time excludes encoding — add encoding time separately if needed
print(f"{'Query':<35} {'ChromaDB (ms)':>15} {'FAISS (ms)':>12}")
print("-" * 65)
for q, ct, ft in zip(query_texts, chroma_times, faiss_times):
    print(f"{q:<35} {ct:>15.2f} {ft:>12.2f}")

print(f"\nAvg ChromaDB: {sum(chroma_times)/len(chroma_times):.2f} ms")
print(f"Avg FAISS:    {sum(faiss_times)/len(faiss_times):.2f} ms")

Query                                 ChromaDB (ms)   FAISS (ms)
-----------------------------------------------------------------
artificial intelligence research              17.88         0.04
space exploration missions                    14.32         0.03
cricket sports results                        14.53         0.02
programming languages                         15.39         0.04
medical science breakthroughs                 15.16         0.02

Avg ChromaDB: 15.46 ms
Avg FAISS:    0.03 ms


In [13]:
print(f"{'Query':<35} {'ChromaDB (ms)':>15} {'FAISS full (ms)':>16}")
print("-" * 70)

chroma_times = []
faiss_full_times = []

for q in query_texts:
    # ChromaDB — encode + search (same as before)
    start = time.perf_counter()
    collection.query(query_texts=[q], n_results=2)
    chroma_times.append((time.perf_counter() - start) * 1000)

    # FAISS — encode + search timed together
    start = time.perf_counter()
    q_vec = model.encode([q]).astype(np.float32)  # now inside timer
    index_flat.search(q_vec, 2)
    faiss_full_times.append((time.perf_counter() - start) * 1000)

for q, ct, ft in zip(query_texts, chroma_times, faiss_full_times):
    print(f"{q:<35} {ct:>15.2f} {ft:>16.2f}")

print(f"\nAvg ChromaDB:    {sum(chroma_times)/len(chroma_times):.2f} ms")
print(f"Avg FAISS full:  {sum(faiss_full_times)/len(faiss_full_times):.2f} ms")

Query                                 ChromaDB (ms)  FAISS full (ms)
----------------------------------------------------------------------
artificial intelligence research              20.32             9.32
space exploration missions                    20.05             8.17
cricket sports results                        13.06             6.19
programming languages                         13.13             6.11
medical science breakthroughs                 13.13             5.97

Avg ChromaDB:    15.94 ms
Avg FAISS full:  7.15 ms
